In [1]:
import os
import pandas as pd
import numpy as np

#### Constants

In [2]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_target = 'pitch_type'

# most common pitch types
list_str_pitch_type = [
    'FF',
    'SL',
    'SI',
    'FT',
    'CH',
    'CU',
    'FC',
    'FS',
]

Bucket: assessment-swish-analytics
Task: 02_data_split


#### Import Data

In [3]:
%%time

str_filename = 'pitches'
str_uri = f's3://{str_bucket}/00_data_collection/{str_filename}'
df = pd.read_csv(str_uri)
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)
<timed exec>:3: DtypeWarning: Columns (29,30) have mixed types. Specify dtype option on import or set low_memory=False.


CPU times: user 9.77 s, sys: 1.64 s, total: 11.4 s
Wall time: 30.2 s


,uid,game_pk,year,date,team_id_b,team_id_p,inning,top,at_bat_num,pcount_at_bat,...,runner7_start,runner7_end,runner7_event,runner7_score,runner7_rbi,runner7_earned,created_at,added_at,modified_at,modified_by
0,14143226,286874,2011,2011-03-31,108,118,1,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
1,14143227,286874,2011,2011-03-31,108,118,1,1,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
2,14143228,286874,2011,2011-03-31,108,118,1,1,1,3,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
3,14143229,286874,2011,2011-03-31,108,118,1,1,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
4,14143230,286874,2011,2011-03-31,108,118,1,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 21:33:20,2016-03-03 21:33:20,2016-03-03 21:33:20,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
718956,19838192,317073,2011,2011-10-28,140,138,9,1,72,3,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718957,19838193,317073,2011,2011-10-28,140,138,9,1,72,4,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718958,19838194,317073,2011,2011-10-28,140,138,9,1,72,5,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1
718959,19838195,317073,2011,2011-10-28,140,138,9,1,73,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2016-03-03 22:23:19,2016-03-03 22:23:19,2016-03-03 22:23:19,1


#### Filter to Main Pitch Types

In [4]:
# drop rows with missing pitch_type
df = df[df[str_target].notna()].copy()
print(f'After dropping missing {str_target}: {df.shape}')

# keep only main pitch types (drops IN, PO, AB, UN, KN, EP, SC, FO, FA)
df = df[df[str_target].isin(list_str_pitch_type)].copy()
print(f'After filtering to main types: {df.shape}')
print(f'\nPitch type distribution:')
print(df[str_target].value_counts())

After dropping missing pitch_type: (716681, 125)
After filtering to main types: (698318, 125)

Pitch type distribution:
pitch_type
FF    238541
SL    109756
SI     87740
FT     81056
CH     72641
CU     56379
FC     41702
FS     10503
Name: count, dtype: int64


#### Sort Chronologically

In [5]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date', 'game_pk', 'at_bat_num', 'pcount_at_bat']).reset_index(drop=True)
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Unique dates: {df["date"].nunique()}')
print(f'Unique games: {df["game_pk"].nunique()}')

Date range: 2011-03-31 00:00:00 to 2011-10-28 00:00:00
Unique dates: 203
Unique games: 2466


#### Out-of-Time Split

Split by date to simulate real-world deployment (predict future pitches from past data).
- **Train**: ~70% (earliest games)
- **Validation**: ~15% (middle games, for hyperparameter tuning)
- **Test**: ~15% (latest games, final evaluation)

In [6]:
# get unique dates sorted
arr_dates = np.sort(df['date'].unique())
n_dates = len(arr_dates)
print(f'Total unique dates: {n_dates}')

# split points
int_train_end = int(n_dates * 0.70)
int_valid_end = int(n_dates * 0.85)

dt_train_cutoff = arr_dates[int_train_end]
dt_valid_cutoff = arr_dates[int_valid_end]

print(f'Train: start to {dt_train_cutoff} ({int_train_end} dates)')
print(f'Valid: {dt_train_cutoff} to {dt_valid_cutoff} ({int_valid_end - int_train_end} dates)')
print(f'Test:  {dt_valid_cutoff} to end ({n_dates - int_valid_end} dates)')

Total unique dates: 203
Train: start to 2011-08-23T00:00:00.000000000 (142 dates)
Valid: 2011-08-23T00:00:00.000000000 to 2011-09-22T00:00:00.000000000 (30 dates)
Test:  2011-09-22T00:00:00.000000000 to end (31 dates)


In [7]:
df_train = df[df['date'] < dt_train_cutoff].copy()
df_valid = df[(df['date'] >= dt_train_cutoff) & (df['date'] < dt_valid_cutoff)].copy()
df_test = df[df['date'] >= dt_valid_cutoff].copy()

print(f'Train: {df_train.shape[0]:,} rows ({df_train.shape[0]/len(df)*100:.1f}%) | {df_train["date"].min()} to {df_train["date"].max()}')
print(f'Valid: {df_valid.shape[0]:,} rows ({df_valid.shape[0]/len(df)*100:.1f}%) | {df_valid["date"].min()} to {df_valid["date"].max()}')
print(f'Test:  {df_test.shape[0]:,} rows ({df_test.shape[0]/len(df)*100:.1f}%) | {df_test["date"].min()} to {df_test["date"].max()}')

Train: 538,294 rows (77.1%) | 2011-03-31 00:00:00 to 2011-08-22 00:00:00
Valid: 120,479 rows (17.3%) | 2011-08-23 00:00:00 to 2011-09-21 00:00:00
Test:  39,545 rows (5.7%) | 2011-09-22 00:00:00 to 2011-10-28 00:00:00


In [8]:
# verify pitch type distribution is similar across splits
df_split_dist = pd.DataFrame({
    'train': df_train[str_target].value_counts(normalize=True) * 100,
    'valid': df_valid[str_target].value_counts(normalize=True) * 100,
    'test': df_test[str_target].value_counts(normalize=True) * 100,
}).round(1)
print('Pitch type distribution (%) across splits:')
df_split_dist

Pitch type distribution (%) across splits:


,train,valid,test
pitch_type,,,
CH,10.5,10.0,10.2
CU,8.0,8.1,8.5
FC,5.9,6.1,6.3
FF,33.9,35.2,34.8
FS,1.5,1.5,1.6
FT,11.5,11.8,11.9
SI,12.9,11.6,11.4
SL,15.8,15.7,15.2


#### Save Splits to S3

In [9]:
for str_name, df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    str_s3_path = f's3://{str_bucket}/{str_task}/{str_name}.csv'
    df_split.to_csv(str_s3_path, index=False)
    print(f'Saved {str_name}.csv to {str_s3_path} ({df_split.shape})')

Saved train.csv to s3://assessment-swish-analytics/02_data_split/train.csv ((538294, 125))
Saved valid.csv to s3://assessment-swish-analytics/02_data_split/valid.csv ((120479, 125))
Saved test.csv to s3://assessment-swish-analytics/02_data_split/test.csv ((39545, 125))
